# Importing and preparing rental apartments data

**This Jupyter notebook does the following:**

- Imports web-scraped rental apartment data of Winterthur from a CSV file
- Inspects the number of rows/columns and the data types
- Extracts the number of rooms, living area and rental price from raw strings using regular expressions (regex)
- Counts missing and duplicated values
- Saves the prepared data to `Data/apartments_data_prepared.csv`

## Libraries and settings

In [1]:
# Libraries
import os
import re
import time
import fnmatch
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print(os.getcwd())

/workspaces/ARM-Stejsnat/Week_08


## Importing data

In [2]:
# Show .csv - files in the folder 'Data'
flist = fnmatch.filter(os.listdir('./Data'), '*.csv')
for i in flist:
    print(i)

# Read the data to a pandas data frame
df = pd.read_csv('./Data/apartments_data_winterthur.csv', sep=',', encoding='utf-8')

# Show first records of data frame
df.head()

apartments_data_winterthur.csv
apartments_data_zuerich.csv


,web-scraper-order,web-scraper-start-url,rooms_area_price_raw,address_raw,price_raw,description_raw,text_raw
0,1693993818-1,https://www.immoscout24.ch/de/wohnung/mieten/o...,"6,5 Zimmer, 143 m², CHF 3017.—","Am Eulachpark 25, 8404 Winterthur, ZH",CHF 3017.—,«Sie suchen die spezielle Maisonettewohnung?»,"6,5 Zimmer, 143 m², CHF 3017.—Am Eulachpark 25..."
1,1693993818-2,https://www.immoscout24.ch/de/wohnung/mieten/o...,"1 Zimmer, 132 m², CHF 3260.—","Katharina Sulzer Platz 2, 8400 Winterthur, ZH",CHF 3260.—,«In Loft-iger Höhe MIETEN OHNE KAUTION»,"1 Zimmer, 132 m², CHF 3260.—Katharina Sulzer P..."
2,1693993818-3,https://www.immoscout24.ch/de/wohnung/mieten/o...,"4,5 Zimmer, 117 m², CHF 3782.—","8400 Winterthur, ZH",CHF 3782.—,"«MÖBLIERT, TEMPORÄR: 4½ ZI-WOHNUNG IN WINTERTH...","4,5 Zimmer, 117 m², CHF 3782.—8400 Winterthur,..."
3,1693993818-4,https://www.immoscout24.ch/de/wohnung/mieten/o...,"3,5 Zimmer, 88 m², CHF 2244.—","Untere Briggerstrasse 66, 8406 Winterthur, ZH",CHF 2244.—,«Modernes Leben im EG mit Gartensitzplatz»,"3,5 Zimmer, 88 m², CHF 2244.—Untere Briggerstr..."
4,1693993818-5,https://www.immoscout24.ch/de/wohnung/mieten/o...,"3,5 Zimmer, 80 m², CHF 1980.—","Wülflingerstrasse 25, 8400 Winterthur, ZH",CHF 1980.—,«Schöne 3.5-Zimmerwohnung mit Balkon zu vermie...,"3,5 Zimmer, 80 m², CHF 1980.—Wülflingerstrasse..."


## Count number of rows and columns in the data frame

In [3]:
# Dimension (rows, columns)
print('Dimension:', df.shape)

# Number of rows
print('Number of rows:', df.shape[0])

# Number of columns
print('Number of columns:', df.shape[1])

Dimension: (120, 7)
Number of rows: 120
Number of columns: 7


## Get data types (raw-format from web scraping)

In [4]:
# Get data types (note that since pandas 3.0, text columns have the data type 'str';
# in older versions of pandas, they have the data type 'object')
df.dtypes

web-scraper-order        str
web-scraper-start-url    str
rooms_area_price_raw     str
address_raw              str
price_raw                str
description_raw          str
text_raw                 str
dtype: object

## Extract and save relevant information from raw data using regular expressions (regex)

### Extract number of rooms

In [5]:
# Extract values from 'rooms_area_price_raw' strings
rooms = []
for i in df['rooms_area_price_raw']:
    d1 = re.findall('(.*)Zimmer', i)
    try:
        d2 = d1[0].strip().replace(',', '.')
    except:
        d2 = None
    rooms.append(d2)

# Save as new variable in the pandas data frame
df['rooms'] = pd.Series(rooms, dtype="float64")
    
# Print first 5 values
print(df['rooms_area_price_raw'].head(5), '\n')
print(df['rooms'].head(5))

0    6,5 Zimmer, 143 m², CHF 3017.—
1      1 Zimmer, 132 m², CHF 3260.—
2    4,5 Zimmer, 117 m², CHF 3782.—
3     3,5 Zimmer, 88 m², CHF 2244.—
4     3,5 Zimmer, 80 m², CHF 1980.—
Name: rooms_area_price_raw, dtype: str 

0    6.5
1    1.0
2    4.5
3    3.5
4    3.5
Name: rooms, dtype: float64


### Extract living area

In [6]:
# Extract values from 'rooms_area_price_raw' strings
area = []
for i in df['rooms_area_price_raw']:
    d1 = re.findall('Zimmer, (.*)m²', i)
    try:
        d2 = d1[0].strip()
    except:
        d2 = None
    area.append(d2)

# Save as new variable in the pandas data frame
df['area'] = pd.Series(area, dtype="Int64")

# Print first 5 values
print(df['rooms_area_price_raw'].head(5), '\n')
print(df['area'].head(5))

0    6,5 Zimmer, 143 m², CHF 3017.—
1      1 Zimmer, 132 m², CHF 3260.—
2    4,5 Zimmer, 117 m², CHF 3782.—
3     3,5 Zimmer, 88 m², CHF 2244.—
4     3,5 Zimmer, 80 m², CHF 1980.—
Name: rooms_area_price_raw, dtype: str 

0    143
1    132
2    117
3     88
4     80
Name: area, dtype: Int64


### Extract rental price

In [7]:
# Extract values from 'price_raw' strings
price = []
for i in df['price_raw']:
    d1 = re.findall('[0-9]+', re.sub(r"[\s']", '', i))  # remove thousands separators, e.g. 'CHF 11 500.-'
    try:
        d2 = d1[0].strip()
    except:
        d2 = None
    price.append(d2)

# Save as new variable in the pandas data frame
df['price'] = pd.Series(price, dtype="Int64")

# Print first 5 values
print(df['price_raw'].head(5), '\n')
print(df['price'].head(5))

0    CHF 3017.—
1    CHF 3260.—
2    CHF 3782.—
3    CHF 2244.—
4    CHF 1980.—
Name: price_raw, dtype: str 

0    3017
1    3260
2    3782
3    2244
4    1980
Name: price, dtype: Int64


### Get data types of all variables including the new ones

In [8]:
df.dtypes

web-scraper-order            str
web-scraper-start-url        str
rooms_area_price_raw         str
address_raw                  str
price_raw                    str
description_raw              str
text_raw                     str
rooms                    float64
area                       Int64
price                      Int64
dtype: object

## Count and identify missing values (if any)

In [9]:
# Count missing values
print(pd.isna(df).sum())

# Identify rows with missing values
df[df.isna().any(axis=1)].head()

web-scraper-order         0
web-scraper-start-url     0
rooms_area_price_raw      0
address_raw               0
price_raw                 0
description_raw           0
text_raw                  0
rooms                     1
area                     11
price                     1
dtype: int64


,web-scraper-order,web-scraper-start-url,rooms_area_price_raw,address_raw,price_raw,description_raw,text_raw,rooms,area,price
6,1693993818-7,https://www.immoscout24.ch/de/wohnung/mieten/o...,"3,5 Zimmer, 171 m², Preis auf Anfrage","Ernst-Jung-Gasse 16B, 8400 Winterthur, ZH",Preis auf Anfrage,«Erstbezug in 3.5 Zimmer-Maisonette-Wohnung in...,"3,5 Zimmer, 171 m², Preis auf AnfrageErnst-Jun...",3.5,171,<NA>
13,1693993818-14,https://www.immoscout24.ch/de/wohnung/mieten/o...,"1 Zimmer, CHF 800.—","Walkestrasse 5, 8400 Winterthur, ZH",CHF 800.—,«Schönes 1 Zimmer Studio (möbliert) im zentral...,"1 Zimmer, CHF 800.—Walkestrasse 5, 8400 Winter...",1.0,<NA>,800
20,1693993818-21,https://www.immoscout24.ch/de/wohnung/mieten/o...,"4,5 Zimmer, CHF 2200.—","Tösstalstrasse 234, 8405 Winterthur, ZH",CHF 2200.—,«4.5 Zimmerwohnung zu vermieten in Winterthur ...,"4,5 Zimmer, CHF 2200.—Tösstalstrasse 234, 8405...",4.5,<NA>,2200
27,1693993822-28,https://www.immoscout24.ch/de/wohnung/mieten/o...,"3 Zimmer, CHF 1640.—","Auwiesenstrasse 13, 8406 Winterthur, ZH",CHF 1640.—,"«Auwiesenstrasse 13, 8406 Winterthur»","3 Zimmer, CHF 1640.—Auwiesenstrasse 13, 8406 W...",3.0,<NA>,1640
34,1693993822-35,https://www.immoscout24.ch/de/wohnung/mieten/o...,"5,5 Zimmer, CHF 2960.—","Schlosstalstrasse 219, 8408 Winterthur, ZH",CHF 2960.—,«Wohnen an der Töss»,"5,5 Zimmer, CHF 2960.—Schlosstalstrasse 219, 8...",5.5,<NA>,2960


## Count and identify duplicated values (if any)

In [10]:
# Columns which identify an apartment listing (the 'web-scraper-order' is unique for
# every scraped row and must therefore not be used to identify duplicated listings)
dup_cols = ['address_raw', 'rooms_area_price_raw', 'price_raw']

# Count duplicated values
print(df.duplicated(subset=dup_cols).sum())

# Identify rows with duplicated values (all occurrences)
df[df.duplicated(subset=dup_cols, keep=False)]

1


,web-scraper-order,web-scraper-start-url,rooms_area_price_raw,address_raw,price_raw,description_raw,text_raw,rooms,area,price
22,1693993818-23,https://www.immoscout24.ch/de/wohnung/mieten/o...,"1,5 Zimmer, 36 m², CHF 2100.—","Zürcherstrasse 143, 8406 Winterthur, ZH",CHF 2100.—,«BUSINESS APARTMENT - Voll möbliert»,"1,5 Zimmer, 36 m², CHF 2100.—Zürcherstrasse 14...",1.5,36,2100
46,1693993822-47,https://www.immoscout24.ch/de/wohnung/mieten/o...,"1,5 Zimmer, 36 m², CHF 2100.—","Zürcherstrasse 143, 8406 Winterthur, ZH",CHF 2100.—,«BUSINESS APARTMENT - FULLY FURNISHED»,"1,5 Zimmer, 36 m², CHF 2100.—Zürcherstrasse 14...",1.5,36,2100


### Ergebnisse der Prüfung

- Fehlende Werte: `rooms` = 1, `area` = 11 und `price` = 1. In den übrigen Spalten fehlen keine Werte.
- Die Zimmerzahl fehlt bei einem Inserat, in dem nur die Fläche angegeben ist. Bei 11 Inseraten ist keine Fläche angegeben. Der fehlende Preis stammt von einem Inserat mit «Preis auf Anfrage».
- Es gibt 1 doppeltes Inserat gemäss den geprüften Merkmalen (`address_raw`, `rooms_area_price_raw` und `price_raw`). Es steht an den DataFrame-Indizes 22 und 46; die Beschreibung ist unterschiedlich, aber die übrigen geprüften Merkmale stimmen überein.
- Vollständig identische Zeilen gibt es nicht (`df.duplicated().sum()` ergibt 0).

### Save data to file

In [11]:
df.to_csv('./Data/apartments_data_prepared.csv', 
          sep=",", 
          encoding='utf-8',
          index=False)

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [12]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-11-07 09:40:04
Python Version: 3.11.18
-----------------------------------
